In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


Part 2 — Classification, Impact Simulation, and Model Interpretability

Cleaning and Feature Engineering

In [2]:
# Load the dataset
df = pd.read_csv("../data/raw/home_credit/application_train.csv")

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

# Top 15 columns with most missing values
df.isnull().sum().sort_values(ascending=False).head(15)

(307511, 122)
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONA

COMMONAREA_AVG              214865
COMMONAREA_MODE             214865
COMMONAREA_MEDI             214865
NONLIVINGAPARTMENTS_MEDI    213514
NONLIVINGAPARTMENTS_MODE    213514
NONLIVINGAPARTMENTS_AVG     213514
FONDKAPREMONT_MODE          210295
LIVINGAPARTMENTS_AVG        210199
LIVINGAPARTMENTS_MEDI       210199
LIVINGAPARTMENTS_MODE       210199
FLOORSMIN_MODE              208642
FLOORSMIN_AVG               208642
FLOORSMIN_MEDI              208642
YEARS_BUILD_AVG             204488
YEARS_BUILD_MODE            204488
dtype: int64

In [3]:
#Drop columns with more than 40% missing values
missing_ratio = df.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.4].index

df = df.drop(columns=cols_to_drop)
print(len(cols_to_drop))

49


In [4]:
#For numerical columns, fill missing values with median
cat_cols = df.select_dtypes(include=['object']).columns

#For categorical columns, fill missing values with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_28936\4232866875.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [5]:
#remove outliers in AMT_INCOME_TOTAL
upper = df['AMT_INCOME_TOTAL'].quantile(0.995)
before = len(df)

df = df[df['AMT_INCOME_TOTAL'] <= upper]

print(before - len(df))

1446


In [6]:
#remove outliers in DAYS_EMPLOYED
before = len(df)
df = df[df['DAYS_EMPLOYED'] != 365243]
print(before - len(df))

55290


In [7]:
#Feature engineering
df['credit_income_ratio'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['annuity_income_ratio'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['credit_term'] = df['AMT_CREDIT'] / df['AMT_ANNUITY']
df['days_employed_ratio'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['income_per_family'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

In [8]:
#One-hot encode categorical variables
df = pd.get_dummies(df, drop_first=True)

In [9]:
#Save curated dataset as parquet
df.to_parquet("../data/curated/home_credit_curated.parquet")